In [26]:
import os
import time
import base64
import requests
import pandas as pd
import jalali_pandas
import plotly.express as px

In [27]:
import os

# توکن را در متغیر محیطی MABNA_TOKEN قرار بده (نه داخل کد)
access_token = os.environ["MABNA_TOKEN"]
# Generate Header: "Basic " + Base64(token + ":")
auth_str = f"{access_token}:"
encoded_token = base64.b64encode(auth_str.encode('ascii')).decode('ascii')
headers = {
    "Authorization": f"Basic {encoded_token}",
    "Accept": "application/json"
}

In [28]:
code = {
    "میلگرد": "SteelRebar  ",
    "گندله سنگ آهن": "IronOrePlt",
    "شمش روی": "ZincIngot",
    "شمش سرب": "LeadIngot   ",
    "گواهی سپرده شمش نقره": "SilverBar   ",
    "مس کاتد": "CopperCthd  ",
    "گواهی سپرده شمش طلا": "GoldBar",
    "گواهی سپرده سکه طلا": "GoldCoin",
    # دلار کد بورسی ندارد؛ با نام و نوع (currency) پیدا می‌شود
    "دلار": "name:دلار",
}

In [29]:
BASE = "https://v1.db.api.mabnadp.com"
COLS = ["date_time", "open_price", "high_price", "low_price", "close_price",
        "real_close_price", "volume", "value", "trade_count"]


def resolve_filter(value):
    """کد بورسی -> instrument.code
    'id:17321'    -> instrument.id مشخص (امن‌ترین راه برای دلار)
    'name:دلار'   -> instrument.id از روی نام دقیق و نوع currency (اگر چند نماد بود هشدار با جزئیات)"""
    value = value.strip()
    if value.startswith("id:"):
        return {"instrument.id": value[3:]}
    if not value.startswith("name:"):
        return {"instrument.code": value}
    nm = value[5:]
    r = requests.get(f"{BASE}/exchange/instruments", headers=headers, timeout=60,
                     params={"name": nm, "type": "currency"})
    r.raise_for_status()
    inst = r.json().get("data", [])
    exact = [i for i in inst if i.get("name") == nm] or inst     # فقط نام دقیق
    if not exact:
        raise ValueError(f"نماد ارزی با نام {nm} پیدا نشد")
    if len(inst) > 1:
        print(f"هشدار: {len(inst)} نماد برای «{nm}» پیدا شد؛ «{exact[0]['id']}» انتخاب شد. جزئیات:")
        for i in inst:
            print("   ", {k: i.get(k) for k in ("id", "code", "name", "english_name", "short_name", "exchange", "market")})
        print("   برای ثابت‌کردن، در دیکشنری code مقدار را به 'id:<شناسه>' تغییر بده.")
    return {"instrument.id": exact[0]["id"]}


def fetch_all_trades(value):
    """کل تاریخچه معاملات روزانه یک نماد (با صفحه‌بندی)"""
    flt = resolve_filter(value)
    rows, skip, size = [], 0, 100
    while True:
        params = {**flt, "_sort": "date_time", "_count": size, "_skip": skip}
        r = requests.get(f"{BASE}/exchange/trades", headers=headers, params=params, timeout=60)
        r.raise_for_status()
        batch = r.json().get("data", [])
        rows += batch
        if len(batch) < size:
            break
        skip += size
    return rows


frames = []
for name, c in code.items():
    c = c.strip()
    rows = fetch_all_trades(c)
    if not rows:
        print(f"{name} ({c}): بدون داده")
        continue
    df = pd.DataFrame(rows)
    df = df[df["date_time"].notna()]                      # رکوردهای حذف‌شده تاریخ ندارند
    df = df[[x for x in COLS if x in df.columns]].copy()
    df["date"] = pd.to_datetime(
        df["date_time"].str[:8].jalali.parse_jalali("%Y%m%d").jalali.to_gregorian(), errors="coerce"
    ).astype("datetime64[ns]")
    df["j_date"] = df["date_time"].str[:4] + "/" + df["date_time"].str[4:6] + "/" + df["date_time"].str[6:8]
    df.insert(0, "name", name)
    df.insert(1, "code", c)
    frames.append(df.drop(columns="date_time"))
    print(f"{name} ({c}): {len(df)} روز  [{df['j_date'].iloc[0]} تا {df['j_date'].iloc[-1]}]")

mabna = pd.concat(frames, ignore_index=True)
mabna = mabna.dropna(subset=["date"])
mabna = mabna[mabna["close_price"] > 0]                   # روزهای بدون معامله
mabna = mabna.drop_duplicates(["name", "date"], keep="last")
mabna = mabna.rename(columns={c: f"mabna_{c}" for c in
                              ["open_price", "high_price", "low_price", "close_price",
                               "real_close_price", "volume", "value", "trade_count"]
                              if c in mabna.columns})
mabna.tail()

میلگرد (SteelRebar): 231 روز  [1404/07/30 تا 1405/07/26]
گندله سنگ آهن (IronOrePlt): 231 روز  [1404/07/30 تا 1405/07/26]
شمش روی (ZincIngot): 231 روز  [1404/07/30 تا 1405/07/26]
شمش سرب (LeadIngot): 72 روز  [1405/04/20 تا 1405/07/26]
گواهی سپرده شمش نقره (SilverBar): 452 روز  [1403/09/20 تا 1405/07/26]
مس کاتد (CopperCthd): 231 روز  [1404/07/30 تا 1405/07/26]
گواهی سپرده شمش طلا (GoldBar): 768 روز  [1402/08/11 تا 1405/07/26]
گواهی سپرده سکه طلا (GoldCoin): 572 روز  [1403/04/09 تا 1405/07/26]
هشدار: 2 نماد برای «دلار» پیدا شد؛ «17321» انتخاب شد. جزئیات:
    {'id': '17321', 'code': '            ', 'name': 'دلار', 'english_name': 'Dollar', 'short_name': 'دلار', 'exchange': {'id': '8', 'meta': {'type': 'exchange.exchange'}}, 'market': {'id': '1', 'meta': {'type': 'exchange.market'}}}
    {'id': '60658', 'code': None, 'name': None, 'english_name': None, 'short_name': None, 'exchange': None, 'market': None}
   برای ثابت‌کردن، در دیکشنری code مقدار را به 'id:<شناسه>' تغییر بده.
دلار (name:دلا

,name,code,mabna_open_price,mabna_high_price,mabna_low_price,mabna_close_price,mabna_real_close_price,mabna_volume,mabna_value,mabna_trade_count,date,j_date
3877,دلار,name:دلار,2601050.0,2606000.0,2568650.0,2577100.0,NaN,0.0,0.0,0.0,2026-10-13,1405/07/21
3878,دلار,name:دلار,2613700.0,2643050.0,2599000.0,2639000.0,NaN,0.0,0.0,0.0,2026-10-14,1405/07/22
3879,دلار,name:دلار,2669000.0,2679000.0,2669000.0,2679000.0,NaN,0.0,0.0,0.0,2026-10-15,1405/07/23
3880,دلار,name:دلار,2652800.0,2658000.0,2633600.0,2638000.0,NaN,0.0,0.0,0.0,2026-10-17,1405/07/25
3881,دلار,name:دلار,2633750.0,2638000.0,2603750.0,2618000.0,NaN,0.0,0.0,0.0,2026-10-18,1405/07/26


## منابع داده (مستندات دقیق)

ترتیب اجرای سلول‌ها: import ← توکن ← `code` ← دریافت مبنا ← این توضیح ← قیمت جهانی (Yahoo/FRED) ← فیوچرز ۱ ساله (Yahoo) ← **SHFE** (میلگرد و روی) ← **SGX** (گندله) ← ضرایب ← پنل.

**بازه زمانی پنل:** `START_DATE = "2025-11-01"` (نوامبر سال پیش) تا آخرین روز موجود (در سلول پنل؛ `None` یعنی کل تاریخچه). فقط خروجی پنل را برش می‌دهد؛ دریافت داده کامل است.

### ۱) قیمت نمادها: مبنا (بورس کالا)
- API: `GET https://v1.db.api.mabnadp.com/exchange/trades` (احراز هویت Basic با `access_token`)، فیلتر `instrument.code=<کد بورسی>` و برای دلار `instrument.id`، مرتب بر اساس `date_time`، صفحه‌بندی ۱۰۰تایی.
- فیلدها: `open_price, high_price, low_price, close_price` (قیمت پایانی با احتساب حجم مبنا)، `real_close_price` (آخرین معامله)، `volume, value, trade_count`. تاریخ `date_time` شمسی است و به میلادی تبدیل می‌شود (`date`) و شمسی در `j_date` می‌ماند.
- پاک‌سازی: رکوردهای حذف‌شده (بدون `date_time`) و روزهای با `close_price ≤ 0` حذف؛ ردیف‌هایی که `close_price` با `real_close_price` بیش از ۲۵٪ فرق دارد با `real_close_price` جایگزین می‌شوند (`close_fixed`).
- کدهای نماد: `SteelRebar, IronOrePlt, ZincIngot, LeadIngot, SilverBar, CopperCthd, GoldBar`.
- **دلار:** `GET /exchange/instruments?name=دلار&type=currency` ← نماد `17321` (نام «دلار»؛ نماد دوم `60658` بدون نام است). *اینکه `17321` همان دلار بازار آزاد است تأیید نشده.* ستون `dollar` = `close_price` همان روز، فقط با تطبیق دقیق تاریخ.

### ۲) قیمت جهانی: مس، نقره، طلا (Yahoo Finance)
- Endpoint **غیررسمی**: `GET https://query1.finance.yahoo.com/v8/finance/chart/<ticker>?period1=0&period2=<now>&interval=1d`؛ فیلد `indicators.quote[0].close`، تاریخ از `timestamp`.
- تیکرها (همه «نزدیک‌ترین فیوچرز»، نه اسپات): `HG=F` مس COMEX (USD/lb)، `GC=F` طلا (USD/oz)، `SI=F` نقره (USD/oz).
- `global_price_type` در پنل = `front futures (spot proxy)`.

### ۳) فیوچرز ۱ ساله: مس، طلا، نقره (Yahoo، قراردادهای ماهانه)
- تیکر هر قرارداد: `<ریشه><کد ماه><دو رقم سال>.CMX` (مثلاً `HGZ27.CMX`؛ کد ماه `FGHJKMNQUVXZ`). ریشه‌ها: `HG`، `GC`، `SI`.
- هر روز دو قرارداد اطراف «روز + ۳۶۵» بر حسب زمان تا سررسید درون‌یابی می‌شوند (`const_maturity`)؛ سررسید ≈ پایان ماه قرارداد (فرض). اگر فقط یک طرف باشد و فاصله ≤ ۶۰ روز، همان قرارداد.
- محدودیت: قراردادهای منقضی‌شده در Yahoo اغلب پاک می‌شوند، پس تاریخچه‌ی قدیمی ناقص است (طلا/نقره حفره‌ی ۲۰۲۵/۰۳–۰۹ و پیش از ۲۰۲۴/۱۱).

### ۴) میلگرد و روی: SHFE (بورس فیوچرز شانگهای)
- فایل عمومی روزانه: `https://www.shfe.cn/data/tradedata/future/dailydata/kx<YYYYMMDD>.dat` (JSON؛ `o_curinstrument`). محصولات `PRODUCTID` = `rb_f` (میلگرد) و `zn_f` (روی)؛ `DELIVERYMONTH` = YYMM؛ قیمت‌ها یوان بر تن.
- **قیمت روز:** `SETTLEMENTPRICE` قرارداد اصلی = بیشترین `OPENINTEREST` همان روز. ستون‌های `global_ticker` و `global_note` فرمول را ثبت می‌کنند.
- **تبدیل به دلار/تن:** ÷ `(1 + SHFE_VAT)` (پیش‌فرض ۱۳٪؛ فرض، تأییدنشده) ÷ یوان بر دلار همان روز از Yahoo `CNY=X` (ستون `global_fx_usdcny`).
- **فیوچرز ۱ ساله:** درون‌یابی ۱۲ ماهه روی قراردادهای ماهانه (سررسید ≈ روز ۱۵ ماه تحویل؛ فرض). دورترین قرارداد حدود ۱۱ ماه جلوتر است. از قیمت تسویه استفاده می‌شود.
- بازار داخلی چین است (نه FOB ترکیه یا LME). *شرایط استفاده/بازنشر داده‌ی SHFE بررسی نشده.*
- اگر SHFE دریافت نشود: روی ← میانگین ماهانه LME/IMF از FRED (`PZINCUSDM`)؛ میلگرد ← `HRC=F` (پروکسی ضعیف).

### ۵) گندله: SGX (بورس سنگاپور)، آهن ۶۵٪ Fe
- API عمومی: `GET https://api.sgx.com/derivatives/v1.0/history/symbol/<SYMBOL>?days=10y&category=futures&params=base-date,daily-settlement-price-abs`؛ تسویه روزانه‌ی هر قرارداد ماهانه، **شامل قراردادهای منقضی‌شده**. سمبل = `M65F` + کد ماه + دو رقم سال (مثلاً `M65FZ26`).
- `M65F` = «SGX MB Iron Ore CFR China (65% Fe Fines) Index Futures»؛ تسویه نقدی بر اساس میانگین ماهانه‌ی شاخص Fastmarkets ۶۵٪ Fe، یعنی پایه‌ی قیمت‌گذاری گندله‌ی Vale (طبق خبرهای Fastmarkets).
- **قیمت روز:** تسویه‌ی نزدیک‌ترین قرارداد (کوچک‌ترین سررسید ≥ امروز؛ سررسید ≈ پایان ماه). **فیوچرز ۱ ساله:** درون‌یابی ۱۲ ماهه روی قراردادها (منحنی تا ~۳ سال جلو).
- **پریمیوم گندله نسبت به شاخص ۶۵٪ در داده نیست.** `PELLET_PREMIUM_USD` (پیش‌فرض ۰) به هر دو قیمت اضافه می‌شود. پریمیوم‌های Vale فصلی و از اخبار قابل جمع‌آوری‌اند (مثلاً مقادیر ۵۰ تا ۶۰ دلار بر تن در گزارش‌های Argus/Platts؛ سال‌ها و ماهیت دقیق را خودت چک کن).
- این API غیررسمی و بدون مستندات است؛ *شرایط استفاده بررسی نشده*. اگر دریافت نشود، `TIO=F` (آهن ۶۲٪، Yahoo) می‌ماند.

### ۶) جایگزین دستی (CSV)
- `global_prices/<نام نماد>.csv` (ستون‌های `date,price`، به واحد جدول) جایگزین قیمت روز؛ `global_prices/<نام نماد>_fut1y.csv` جایگزین فیوچرز ۱ ساله. نوع قیمت `spot (csv)` ثبت می‌شود.
- آنچه منبع ندارد: سرب (قیمت جهانی و فیوچرز).

### ۷) تأخیر و تطبیق تاریخ
- قیمت جهانی هر ردیف = آخرین قیمت جهانی تا (روز − `GLOBAL_LAG_DAYS`=۱)، حداکثر ۷ روز قبل. تاریخ واقعی در `global_date` و `global_fut_1y_date`.

### ۸) ضرایب تبدیل
- از اطلاعیه‌های بورس کالا: مس/روی/گندله/میلگرد ۱ کیلوگرم، نقره ۱ گرم، طلا ۰٫۱ گرم (قیمت به ازای هر ۰٫۱ گرم). جدول `FACTORS` در سلول مربوطه.

### ۹) بررسی سلامت و نوع سیگنال
- سلول «حباب» جدول `sanity` را چاپ می‌کند: نسبت دلار ذاتی به دلار بازار (میانه و آخرین روز). نسبت خارج از ۰٫۴ تا ۲٫۵ پرچم «⚠ واحد/مرجع را چک کن» می‌گیرد. این همان خطایی است که قبلاً دلار ذاتی طلا را ۱۰ برابر کوچک (فرض ۱ گرم به‌جای ۰٫۱ گرم) و میلگرد با `HRC=F` را حدود ۷۲٪ پایین نشان می‌داد.
- نمودارهای `implied_dollar` (دلار ذاتی در برابر دلار بازار) و `implied_dollar_bubble` هر بار از `panel` ساخته می‌شوند؛ در صورت نبود `nbformat` به‌صورت `implied_dollar.html` و `implied_dollar_bubble.html` ذخیره می‌شوند. فایل‌های HTML قدیمی که با نسخه‌های قبلی ساخته شده‌اند معتبر نیستند.
- برچسب `signal_type` (global-anchored، self-referenced) و شواهدش: سلول «شواهد» و `docs/signal_evidence.md`.


In [30]:
# price_type: "front futures (spot proxy)" = نزدیک‌ترین فیوچرز به‌عنوان جایگزین اسپات (اسپات رایگان نداریم)
GLOBAL_SPEC = {
    "مس کاتد":               dict(ticker="HG=F",  unit="USD/lb",        price_type="front futures (spot proxy)"),
    # ZNC=F در Yahoo عملاً قیمت جدید نمی‌دهد (ماه‌ها ثابت)؛ به‌جایش میانگین ماهانه LME (IMF) از FRED
    # (روزانه نیست و ممکن است دیرتر منتشر شود). برای قیمت روزانه فایل CSV بگذار.
    "شمش روی":               dict(ticker="FRED:PZINCUSDM", unit="USD/t", price_type="monthly avg (FRED/IMF)"),
    "گندله سنگ آهن":         dict(ticker="TIO=F", unit="USD/t",         price_type="front futures (spot proxy)"),
    "میلگرد":                dict(ticker="HRC=F", unit="USD/short ton", price_type="front futures (proxy)"),
    "گواهی سپرده شمش نقره":  dict(ticker="SI=F",  unit="USD/oz",        price_type="front futures (spot proxy)"),
    "گواهی سپرده شمش طلا":   dict(ticker="GC=F",  unit="USD/oz",        price_type="front futures (spot proxy)"),
    "گواهی سپرده سکه طلا": dict(ticker="GC=F",  unit="USD/oz",        price_type="front futures (spot proxy)")
}


def fetch_yahoo(ticker):
    r = requests.get(
        f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}",
        params={"period1": 0, "period2": int(time.time()) + 86400, "interval": "1d"},
        headers={"User-Agent": "Mozilla/5.0"}, timeout=60,
    )
    r.raise_for_status()
    res = r.json()["chart"]["result"][0]
    s = pd.Series(res["indicators"]["quote"][0]["close"],
                  index=pd.to_datetime(res["timestamp"], unit="s").normalize(), dtype="float64")
    return s[~s.index.duplicated(keep="last")].dropna().sort_index()


def fetch_fred_monthly_as_daily(series_id):
    """سری ماهانه FRED -> روزانه (مقدار هر ماه برای همه روزهای همان ماه)"""
    d = pd.read_csv(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}")
    d.columns = ["date", "price"]
    d["price"] = pd.to_numeric(d["price"], errors="coerce")
    s = pd.Series(d["price"].values, index=pd.to_datetime(d["date"])).dropna().sort_index()
    end = max(s.index[-1] + pd.offsets.MonthEnd(0), pd.Timestamp.today().normalize())
    return s.reindex(pd.date_range(s.index[0], end, freq="D")).ffill()


def report_staleness(name, s):
    """هشدار برای سری‌هایی که قیمتشان به‌روز نمی‌شود"""
    same = (s.diff() == 0).mean()
    if same > 0.5:
        print(f"  ⚠ {name}: {same:.0%} روزها بدون تغییر است؛ منبع احتمالاً راکد است")


glob_frames = []
for name, spec in GLOBAL_SPEC.items():
    path = f"global_prices/{name}.csv"
    try:
        if os.path.exists(path):
            d = pd.read_csv(path)
            s = pd.Series(d["price"].values, index=pd.to_datetime(d["date"]), dtype="float64")
            s = s[~s.index.duplicated(keep="last")].dropna().sort_index()
            spec["price_type"] = "spot (csv)"
        elif spec["ticker"].startswith("FRED:"):
            s = fetch_fred_monthly_as_daily(spec["ticker"][5:])
        else:
            s = fetch_yahoo(spec["ticker"])
    except Exception as e:
        print(f"{name}: خطا -> {e}  (قیمت جهانی این نماد خالی می‌ماند)")
        continue
    g = s.rename("global_price").rename_axis("date").reset_index()
    g["date"] = pd.to_datetime(g["date"]).astype("datetime64[ns]")
    g["name"] = name
    g["global_ticker"] = spec["ticker"] if spec["price_type"] != "spot (csv)" else path
    g["global_unit"] = spec["unit"]
    g["global_price_type"] = spec["price_type"]
    glob_frames.append(g)
    print(f"{name} [{spec['ticker']}, {spec['price_type']}]: {len(g)} روز  {g['date'].min().date()} تا {g['date'].max().date()}")
    report_staleness(name, s)

global_df = pd.concat(glob_frames, ignore_index=True)

مس کاتد [HG=F, front futures (spot proxy)]: 6562 روز  2000-08-30 تا 2026-10-16
شمش روی [FRED:PZINCUSDM, monthly avg (FRED/IMF)]: 12710 روز  1992-01-01 تا 2026-10-18
  ⚠ شمش روی: 97% روزها بدون تغییر است؛ منبع احتمالاً راکد است
گندله سنگ آهن [TIO=F, front futures (spot proxy)]: 4013 روز  2010-10-14 تا 2026-10-16
میلگرد [HRC=F, front futures (proxy)]: 4487 روز  2008-10-20 تا 2026-10-16
گواهی سپرده شمش نقره [SI=F, front futures (spot proxy)]: 6559 روز  2000-08-30 تا 2026-10-16
گواهی سپرده شمش طلا [GC=F, front futures (spot proxy)]: 6558 روز  2000-08-30 تا 2026-10-16
گواهی سپرده سکه طلا [GC=F, front futures (spot proxy)]: 6558 روز  2000-08-30 تا 2026-10-16


In [31]:
import numpy as np
from concurrent.futures import ThreadPoolExecutor

# ============ فیوچرز یک‌ساله (ثابت‌-سررسید ۱۲ ماهه) ============
# برای هر روز، دو قرارداد اطراف «امروز + ۱ سال» را از Yahoo می‌گیریم و بر اساس زمان تا سررسید درون‌یابی می‌کنیم.
# سررسید هر قرارداد تقریباً «پایان ماه قرارداد» فرض شده است (خطای چند روزه).
# برای TIO (آهن) و روی، قراردادهای جداگانه در Yahoo نیست؛ برای آن‌ها فایل
# global_prices/<نام نماد>_fut1y.csv  با ستون‌های date,price (همان واحد قیمت اسپات) بگذار.
FUT_SPEC = {            # نماد -> (ریشه قرارداد، پسوند بورس در Yahoo)
    "مس کاتد":               ("HG",  "CMX"),
    "گواهی سپرده شمش طلا":   ("GC",  "CMX"),
    "گواهی سپرده سکه طلا":   ("GC",  "CMX"),
    "گواهی سپرده شمش نقره":  ("SI",  "CMX"),
    "میلگرد":                ("HRC", "CMX"),     # پروکسی (ورق گرم آمریکا)
}
HORIZON_DAYS = 365
MAX_GAP_DAYS = 60       # حداکثر فاصله مجاز سررسید قرارداد تا هدف وقتی درون‌یابی ممکن نیست
FUT_START = pd.Timestamp("2023-06-01")                      # قراردادهای منقضی‌شده قبل از این لازم نیست
FUT_END   = pd.Timestamp.today().normalize() + pd.DateOffset(months=30)
CACHE_DIR = "cache/futures"                                 # کش روزانه؛ برای دانلود دوباره پاکش کن
MONTH_CODE = dict(zip(range(1, 13), "FGHJKMNQUVXZ"))
os.makedirs(CACHE_DIR, exist_ok=True)


def contract_list(root, exch):
    out = []
    for ts in pd.date_range(FUT_START.replace(day=1), FUT_END, freq="MS"):
        out.append((f"{root}{MONTH_CODE[ts.month]}{str(ts.year)[2:]}.{exch}", ts + pd.offsets.MonthEnd(0)))
    return out


def fetch_contract(ticker):
    """قیمت روزانه یک قرارداد (با کش یک‌روزه)؛ اگر در Yahoo نبود None.
    فقط «قرارداد وجود ندارد» (۴۰۴) کش می‌شود؛ خطای موقت کش نمی‌شود."""
    path = f"{CACHE_DIR}/{ticker}.csv"
    if os.path.exists(path) and time.time() - os.path.getmtime(path) < 86400:
        d = pd.read_csv(path, parse_dates=["date"])
        return ticker, (d.set_index("date")["price"] if len(d) else None)
    try:
        s = fetch_yahoo(ticker)
    except requests.HTTPError as e:
        if e.response is not None and e.response.status_code == 404:
            pd.DataFrame({"date": [], "price": []}).to_csv(path, index=False)
        return ticker, None
    except Exception:
        return ticker, None
    pd.DataFrame({"date": s.index, "price": s.values}).to_csv(path, index=False)
    return ticker, s


def const_maturity(contracts):
    """contracts: لیست (سررسید، سری قیمت). خروجی: سری قیمت ثابت‌-سررسید HORIZON_DAYS روزه"""
    frame = pd.concat([s for _, s in contracts], axis=1)
    frame.columns = pd.DatetimeIndex([e for e, _ in contracts])
    res = {}
    for d, row in frame.iterrows():
        row = row.dropna()
        if row.empty:
            continue
        target = d + pd.Timedelta(days=HORIZON_DAYS)
        exp = row.index
        lower, upper = exp[exp <= target], exp[exp > target]
        if len(lower) and len(upper):
            e1, e2 = lower.max(), upper.min()
            if (e2 - e1).days <= 2 * MAX_GAP_DAYS:
                w = (target - e1).days / (e2 - e1).days
                res[d] = row[e1] * (1 - w) + row[e2] * w
                continue
        near = exp[np.argmin(np.abs((exp - target).days))]
        if abs((near - target).days) <= MAX_GAP_DAYS:
            res[d] = row[near]
    return pd.Series(res, dtype="float64").sort_index()


fut_frames = []
for name in GLOBAL_SPEC:
    path = f"global_prices/{name}_fut1y.csv"
    if os.path.exists(path):
        d = pd.read_csv(path)
        s = pd.Series(d["price"].values, index=pd.to_datetime(d["date"]), dtype="float64").sort_index()
        src = path
    elif name in FUT_SPEC:
        root, exch = FUT_SPEC[name]
        cons = contract_list(root, exch)
        with ThreadPoolExecutor(8) as ex:
            fetched = dict(ex.map(lambda c: fetch_contract(c[0]), cons))
        contracts = []
        for t, exp in cons:
            ser = fetched.get(t)
            if ser is not None and len(ser):
                contracts.append((exp, ser[ser.index <= exp]))
        if not contracts:
            print(f"{name}: هیچ قراردادی پیدا نشد")
            continue
        s = const_maturity(contracts)
        src = f"{root} {HORIZON_DAYS}d constant-maturity ({len(contracts)} قرارداد Yahoo/{exch})"
    else:
        print(f"{name}: فیوچرز یک‌ساله رایگان ندارد -> ستون خالی (می‌توانی {path} بگذاری)")
        continue
    f = s.rename("global_fut_1y").rename_axis("date").reset_index()
    f["date"] = pd.to_datetime(f["date"]).astype("datetime64[ns]")
    f["name"] = name
    f["global_fut_1y_source"] = src
    fut_frames.append(f)
    print(f"{name}: {len(f)} روز  {f['date'].min().date()} تا {f['date'].max().date()}  | آخرین فیوچرز ۱ ساله: {s.iloc[-1]:.4g}")

fut1y_df = (pd.concat(fut_frames, ignore_index=True) if fut_frames
            else pd.DataFrame(columns=["date", "global_fut_1y", "name", "global_fut_1y_source"]))
fut1y_df.tail()

C:\Users\Hosein\AppData\Local\Temp\ipykernel_8904\3636943440.py:53: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  frame = pd.concat([s for _, s in contracts], axis=1)


مس کاتد: 388 روز  2024-10-01 تا 2026-10-16  | آخرین فیوچرز ۱ ساله: 6.896
شمش روی: فیوچرز یک‌ساله رایگان ندارد -> ستون خالی (می‌توانی global_prices/شمش روی_fut1y.csv بگذاری)
گندله سنگ آهن: فیوچرز یک‌ساله رایگان ندارد -> ستون خالی (می‌توانی global_prices/گندله سنگ آهن_fut1y.csv بگذاری)
میلگرد: 284 روز  2025-09-02 تا 2026-10-16  | آخرین فیوچرز ۱ ساله: 1136


C:\Users\Hosein\AppData\Local\Temp\ipykernel_8904\3636943440.py:53: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  frame = pd.concat([s for _, s in contracts], axis=1)


گواهی سپرده شمش نقره: 365 روز  2024-11-01 تا 2026-10-16  | آخرین فیوچرز ۱ ساله: 59.7


C:\Users\Hosein\AppData\Local\Temp\ipykernel_8904\3636943440.py:53: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  frame = pd.concat([s for _, s in contracts], axis=1)


گواهی سپرده شمش طلا: 365 روز  2024-11-01 تا 2026-10-16  | آخرین فیوچرز ۱ ساله: 4350


C:\Users\Hosein\AppData\Local\Temp\ipykernel_8904\3636943440.py:53: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  frame = pd.concat([s for _, s in contracts], axis=1)


گواهی سپرده سکه طلا: 365 روز  2024-11-01 تا 2026-10-16  | آخرین فیوچرز ۱ ساله: 4350


,date,global_fut_1y,name,global_fut_1y_source
1762,2026-10-12,4360.438792,گواهی سپرده سکه طلا,GC 365d constant-maturity (26 قرارداد Yahoo/CMX)
1763,2026-10-13,4348.183799,گواهی سپرده سکه طلا,GC 365d constant-maturity (26 قرارداد Yahoo/CMX)
1764,2026-10-14,4277.400044,گواهی سپرده سکه طلا,GC 365d constant-maturity (26 قرارداد Yahoo/CMX)
1765,2026-10-15,4341.051711,گواهی سپرده سکه طلا,GC 365d constant-maturity (26 قرارداد Yahoo/CMX)
1766,2026-10-16,4350.138562,گواهی سپرده سکه طلا,GC 365d constant-maturity (26 قرارداد Yahoo/CMX)


In [32]:
import json

# ============ SHFE (بورس فیوچرز شانگهای): میلگرد (rb) و روی (zn) ============
# منبع: فایل روزانه عمومی خود SHFE
#   https://www.shfe.cn/data/tradedata/future/dailydata/kx<YYYYMMDD>.dat   (JSON، همه قراردادها)
# فیلدهای استفاده‌شده: PRODUCTID ('rb_f' / 'zn_f')، DELIVERYMONTH (YYMM)، SETTLEMENTPRICE، OPENINTEREST
# قیمت‌ها به یوان بر تن‌اند و (طبق فرض این نوت‌بوک) شامل ۱۳٪ مالیات ارزش افزوده.
#   قیمت روز      = تسویه قرارداد اصلی (بیشترین open interest همان روز)
#   فیوچرز ۱ ساله = درون‌یابی ۱۲ ماهه روی قراردادهای ماهانه (سررسید ≈ روز ۱۵ ماه تحویل؛ فرض)
#   تبدیل         : USD/t = یوان/تن ÷ (1+SHFE_VAT) ÷ (یوان بر دلار همان روز از Yahoo: CNY=X)
SHFE_URL = "https://www.shfe.cn/data/tradedata/future/dailydata/kx{d}.dat"
SHFE_PRODUCTS = {"میلگرد": "rb_f", "شمش روی": "zn_f"}      # نماد -> PRODUCTID در فایل SHFE
SHFE_VAT = 0.13          # 0.0 یعنی بدون کسر مالیات
SHFE_CACHE = "cache/shfe"
os.makedirs(SHFE_CACHE, exist_ok=True)


def shfe_day(day):
    """رکوردهای rb/zn یک روز؛ [] اگر تعطیل بود (۴۰۴)؛ None در خطای موقت"""
    d = day.strftime("%Y%m%d")
    path = f"{SHFE_CACHE}/{d}.json"
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    try:
        r = requests.get(SHFE_URL.format(d=d), headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    except Exception:
        return None
    if r.status_code == 404:
        out = []
    elif r.status_code != 200:
        return None
    else:
        try:
            data = json.loads(r.content.decode("utf-8-sig"))
        except Exception:
            return None
        wanted = set(SHFE_PRODUCTS.values())
        out = []
        for x in data.get("o_curinstrument", []):
            pid = str(x.get("PRODUCTID", "")).strip()
            mth = str(x.get("DELIVERYMONTH", "")).strip()
            if pid in wanted and mth.isdigit():
                out.append(dict(product=pid, month=mth, settle=x.get("SETTLEMENTPRICE"),
                                close=x.get("CLOSEPRICE"), oi=x.get("OPENINTEREST"), volume=x.get("VOLUME")))
    if (pd.Timestamp.today().normalize() - day).days >= 3:      # روزهای اخیر (شاید هنوز منتشر نشده) کش نشود
        with open(path, "w", encoding="utf-8") as f:
            json.dump(out, f)
    return out


shfe_names = [n for n in SHFE_PRODUCTS if n in set(mabna["name"])]
start = mabna.loc[mabna["name"].isin(shfe_names), "date"].min() - pd.Timedelta(days=20)
days = [d for d in pd.date_range(start, pd.Timestamp.today().normalize()) if d.weekday() < 5]
with ThreadPoolExecutor(8) as ex:
    res = list(ex.map(shfe_day, days))
failed = [d.date() for d, r in zip(days, res) if r is None]
if failed:
    print(f"⚠ {len(failed)} روز SHFE دریافت نشد (خطای موقت؛ دوباره اجرا کن):", failed[:5], "...")

shfe = pd.DataFrame([dict(date=d, **x) for d, r in zip(days, res) if r for x in r])
if shfe.empty:
    raise RuntimeError("داده SHFE دریافت نشد؛ اتصال را چک کن (منبع قبلی Yahoo/FRED برای میلگرد و روی می‌ماند).")
shfe["date"] = pd.to_datetime(shfe["date"]).astype("datetime64[ns]")
for c in ["settle", "close", "oi", "volume"]:
    shfe[c] = pd.to_numeric(shfe[c], errors="coerce")
shfe["expiry"] = pd.to_datetime("20" + shfe["month"] + "15", format="%Y%m%d")      # فرض: سررسید روز ۱۵ ماه تحویل

fx = fetch_yahoo("CNY=X")                                   # یوان به‌ازای هر دلار
fx.index = fx.index.astype("datetime64[ns]")
fx = fx.reindex(fx.index.union(pd.DatetimeIndex(days).astype("datetime64[ns]"))).sort_index().ffill()

new_g, new_f, done = [], [], []
for name, pid in SHFE_PRODUCTS.items():
    sp = shfe[(shfe["product"] == pid) & (shfe["settle"] > 0)]
    if sp.empty:
        print(f"{name}: داده SHFE پیدا نشد؛ منبع قبلی (Yahoo/FRED) می‌ماند")
        continue
    # --- قیمت روز: قرارداد اصلی (بیشترین open interest) ---
    main = sp.sort_values(["date", "oi"]).groupby("date").tail(1).set_index("date").sort_index()
    fx_d = fx.reindex(main.index)
    g = pd.DataFrame({
        "date": main.index,
        "global_price": (main["settle"] / (1 + SHFE_VAT) / fx_d).values,
        "name": name,
        "global_ticker": ("SHFE " + pid[:2] + main["month"] + " (main, settlement)").values,
        "global_unit": "USD/t",
        "global_price_type": "SHFE main contract",
        "global_note": f"SHFE {pid} SETTLEMENTPRICE CNY/t / (1+{SHFE_VAT}) / USDCNY (Yahoo CNY=X)",
        "global_fx_usdcny": fx_d.values,
    }).dropna(subset=["global_price"])
    new_g.append(g)
    # --- فیوچرز ۱ ساله: درون‌یابی ۱۲ ماهه روی قراردادهای ماهانه ---
    contracts = []
    for month, grp in sp.groupby("month"):
        ser = grp.drop_duplicates("date").set_index("date")["settle"].sort_index()
        ser = (ser / (1 + SHFE_VAT) / fx.reindex(ser.index)).dropna()
        if len(ser):
            contracts.append((pd.Timestamp(grp["expiry"].iloc[0]), ser))
    s12 = const_maturity(contracts)
    f = s12.rename("global_fut_1y").rename_axis("date").reset_index()
    f["date"] = pd.to_datetime(f["date"]).astype("datetime64[ns]")
    f["name"] = name
    f["global_fut_1y_source"] = f"SHFE {pid} {HORIZON_DAYS}d constant-maturity ({len(contracts)} contracts, settlement, USD/t)"
    new_f.append(f)
    done.append(name)
    GLOBAL_SPEC[name].update(ticker=f"SHFE:{pid}", unit="USD/t", price_type="SHFE main contract")
    rolls = int((main["month"] != main["month"].shift()).sum() - 1)
    print(f"{name} [SHFE {pid}]: {len(g)} روز {g['date'].min().date()} تا {g['date'].max().date()} | آخرین: {g['global_price'].iloc[-1]:.1f} USD/t "
          f"(قرارداد {main['month'].iloc[-1]}، {rolls} بار تعویض قرارداد اصلی) | فیوچرز ۱ ساله: {len(f)} روز، آخرین {s12.iloc[-1]:.1f}")

global_df = pd.concat([global_df[~global_df["name"].isin(done)]] + new_g, ignore_index=True)
fut1y_df = pd.concat([fut1y_df[~fut1y_df["name"].isin(done)]] + new_f, ignore_index=True)
global_df[global_df["name"].isin(done)].groupby("name").tail(2)

میلگرد [SHFE rb_f]: 248 روز 2025-10-09 تا 2026-10-16 | آخرین: 406.3 USD/t (قرارداد 2701، 3 بار تعویض قرارداد اصلی) | فیوچرز ۱ ساله: 248 روز، آخرین 411.3
شمش روی [SHFE zn_f]: 248 روز 2025-10-09 تا 2026-10-16 | آخرین: 3358.8 USD/t (قرارداد 2612، 13 بار تعویض قرارداد اصلی) | فیوچرز ۱ ساله: 248 روز، آخرین 3340.3


,date,global_price,name,global_ticker,global_unit,global_price_type,global_note,global_fx_usdcny
30496,2026-10-15,405.517105,میلگرد,"SHFE rb2701 (main, settlement)",USD/t,SHFE main contract,SHFE rb_f SETTLEMENTPRICE CNY/t / (1+0.13) / U...,6.6909
30497,2026-10-16,406.310681,میلگرد,"SHFE rb2701 (main, settlement)",USD/t,SHFE main contract,SHFE rb_f SETTLEMENTPRICE CNY/t / (1+0.13) / U...,6.6909
30744,2026-10-15,3379.970521,شمش روی,"SHFE zn2612 (main, settlement)",USD/t,SHFE main contract,SHFE zn_f SETTLEMENTPRICE CNY/t / (1+0.13) / U...,6.6909
30745,2026-10-16,3358.808506,شمش روی,"SHFE zn2612 (main, settlement)",USD/t,SHFE main contract,SHFE zn_f SETTLEMENTPRICE CNY/t / (1+0.13) / U...,6.6909


In [33]:
# ============ SGX (بورس سنگاپور): گندله سنگ آهن ← آهن ۶۵٪ Fe (پایه قیمت‌گذاری گندله) ============
# منبع: API عمومی SGX، تسویه روزانه هر قرارداد ماهانه (شامل قراردادهای منقضی‌شده):
#   https://api.sgx.com/derivatives/v1.0/history/symbol/<SYMBOL>?days=10y&category=futures&params=base-date,daily-settlement-price-abs
# محصول M65F = «SGX MB Iron Ore CFR China (65% Fe Fines) Index Futures» (تسویه نقدی بر اساس میانگین ماهانه شاخص Fastmarkets)
#   قیمت روز      = تسویه نزدیک‌ترین قرارداد (کوچک‌ترین سررسید ≥ امروز؛ سررسید ≈ پایان ماه قرارداد)
#   فیوچرز ۱ ساله = درون‌یابی ۱۲ ماهه روی قراردادهای ماهانه
#   پریمیوم گندله نسبت به شاخص ۶۵٪ در این داده نیست؛ PELLET_PREMIUM_USD (دلار/تن) به هر دو قیمت اضافه می‌شود.
SGX_API = "https://api.sgx.com/derivatives/v1.0/history/symbol/{sym}"
SGX_PRODUCTS = {"گندله سنگ آهن": "M65F"}          # نماد -> ریشه قرارداد در SGX
PELLET_PREMIUM_USD = 0.0                         # 0.0 یعنی فقط شاخص ۶۵٪ (بدون پریمیوم گندله)
SGX_CACHE = "cache/sgx"
os.makedirs(SGX_CACHE, exist_ok=True)


def sgx_contract(sym):
    """تسویه روزانه یک قرارداد SGX (کش یک‌روزه)؛ سری خالی اگر قرارداد وجود نداشت؛ None در خطای موقت"""
    path = f"{SGX_CACHE}/{sym}.csv"
    if os.path.exists(path) and time.time() - os.path.getmtime(path) < 86400:
        d = pd.read_csv(path, parse_dates=["date"])
        return sym, d.set_index("date")["price"]
    try:
        r = requests.get(SGX_API.format(sym=sym), timeout=60, headers={"User-Agent": "Mozilla/5.0"},
                         params={"days": "10y", "category": "futures", "params": "base-date,daily-settlement-price-abs"})
        r.raise_for_status()
        data = r.json().get("data") or []
    except Exception:
        return sym, None
    ser = pd.Series({pd.Timestamp(x["base-date"]): float(x["daily-settlement-price-abs"])
                     for x in data if x.get("daily-settlement-price-abs")}, dtype="float64").sort_index()
    ser.rename_axis("date").rename("price").reset_index().to_csv(path, index=False)
    return sym, ser


sgx_new_g, sgx_new_f, sgx_done = [], [], []
for name, root in SGX_PRODUCTS.items():
    if name not in set(mabna["name"]):
        continue
    first = mabna.loc[mabna["name"] == name, "date"].min() - pd.Timedelta(days=40)
    cons = [(f"{root}{MONTH_CODE[ts.month]}{str(ts.year)[2:]}", ts + pd.offsets.MonthEnd(0))
            for ts in pd.date_range(first.replace(day=1), pd.Timestamp.today().normalize() + pd.DateOffset(months=36), freq="MS")]
    with ThreadPoolExecutor(8) as ex:
        fetched = dict(ex.map(lambda c: sgx_contract(c[0]), cons))
    failed = [s for s, v in fetched.items() if v is None]
    if failed:
        print(f"⚠ {len(failed)} قرارداد SGX دریافت نشد (خطای موقت):", failed[:5])
    contracts = [(exp, fetched[s][fetched[s].index <= exp]) for s, exp in cons
                 if fetched.get(s) is not None and len(fetched[s])]
    if not contracts:
        print(f"{name}: داده SGX پیدا نشد؛ منبع قبلی (Yahoo TIO=F) می‌ماند")
        continue
    frame = pd.concat([s for _, s in contracts], axis=1, sort=True)
    frame.columns = pd.DatetimeIndex([e for e, _ in contracts])
    spot = {}
    for d, row in frame.iterrows():
        alive = row.dropna()
        alive = alive[alive.index >= d]
        if len(alive):
            spot[d] = alive.sort_index().iloc[0]               # نزدیک‌ترین سررسید
    spot = pd.Series(spot, dtype="float64").sort_index() + PELLET_PREMIUM_USD
    g = spot.rename("global_price").rename_axis("date").reset_index()
    g["date"] = pd.to_datetime(g["date"]).astype("datetime64[ns]")
    g["name"] = name
    g["global_ticker"] = f"SGX {root} front month"
    g["global_unit"] = "USD/t"
    g["global_price_type"] = "SGX front month (65% Fe index, no pellet premium)" if PELLET_PREMIUM_USD == 0 else "SGX front month + pellet premium"
    g["global_note"] = f"SGX {root} daily settlement (nearest expiry) + premium {PELLET_PREMIUM_USD} USD/t"
    sgx_new_g.append(g)
    s12 = const_maturity(contracts) + PELLET_PREMIUM_USD
    f = s12.rename("global_fut_1y").rename_axis("date").reset_index()
    f["date"] = pd.to_datetime(f["date"]).astype("datetime64[ns]")
    f["name"] = name
    f["global_fut_1y_source"] = f"SGX {root} {HORIZON_DAYS}d constant-maturity ({len(contracts)} contracts, daily settlement, USD/t)"
    sgx_new_f.append(f)
    sgx_done.append(name)
    GLOBAL_SPEC[name].update(ticker=f"SGX:{root}", unit="USD/t", price_type=g["global_price_type"].iloc[0])
    print(f"{name} [SGX {root}]: قیمت روز {len(g)} روز {g['date'].min().date()} تا {g['date'].max().date()} | آخرین {spot.iloc[-1]:.2f} USD/t "
          f"| فیوچرز ۱ ساله {len(f)} روز، آخرین {s12.iloc[-1]:.2f} | قراردادها: {len(contracts)}")

global_df = pd.concat([global_df[~global_df["name"].isin(sgx_done)]] + sgx_new_g, ignore_index=True)
fut1y_df = pd.concat([fut1y_df[~fut1y_df["name"].isin(sgx_done)]] + sgx_new_f, ignore_index=True)
global_df[global_df["name"].isin(sgx_done)].groupby("name").tail(2)

گندله سنگ آهن [SGX M65F]: قیمت روز 980 روز 2023-01-03 تا 2026-10-16 | آخرین 105.93 USD/t | فیوچرز ۱ ساله 573 روز، آخرین 105.96 | قراردادها: 40


,date,global_price,name,global_ticker,global_unit,global_price_type,global_note,global_fx_usdcny
27711,2026-10-15,105.23,گندله سنگ آهن,SGX M65F front month,USD/t,"SGX front month (65% Fe index, no pellet premium)",SGX M65F daily settlement (nearest expiry) + p...,NaN
27712,2026-10-16,105.93,گندله سنگ آهن,SGX M65F front month,USD/t,"SGX front month (65% Fe index, no pellet premium)",SGX M65F daily settlement (nearest expiry) + p...,NaN


In [ ]:
# ================= ضرایب تبدیل: گواهی بورس کالا <-> واحد قیمت جهانی =================
# قیمت جهانی به ازای هر یک گواهی  =  global_price × global_units_per_cert
#   global_units_per_cert = cert_qty × (مقدار «واحد جهانی» در هر «واحد پایه» گواهی)
# توجه: purity فقط ثبت شده و در ضریب اعمال نشده است.

# ۱) ثابت‌های تبدیل واحد
LB_PER_KG        = 2.20462262          # پوند در هر کیلوگرم
TON_PER_KG       = 1 / 1000            # تن متریک در هر کیلوگرم
SHORT_TON_PER_KG = 1 / 907.18474       # short ton در هر کیلوگرم
OZ_PER_G         = 1 / 31.1034768      # اونس تروی در هر گرم

# ۲) واحد جهانی هر نماد  ->  (واحد پایه گواهی، مقدار واحد جهانی در هر واحد پایه)
GLOBAL_UNIT_PER_BASE = {
    "USD/lb":        ("kg", LB_PER_KG),
    "USD/t":         ("kg", TON_PER_KG),
    "USD/short ton": ("kg", SHORT_TON_PER_KG),
    "USD/oz":        ("g",  OZ_PER_G),
}

# ۳) مشخصات گواهی هر نماد (از اطلاعیه‌های بورس کالا)
#    cert_qty: اندازه هر گواهی (به cert_unit)   purity: عیار (نسبت، در صورت وجود)
CERT_SPEC = {
    "مس کاتد":               dict(ime_symbol="CopperCthd", cert_qty=1.0, cert_unit="kg", purity=None),
    "شمش روی":               dict(ime_symbol="ZincIngot",  cert_qty=1.0, cert_unit="kg", purity=None),
    "گندله سنگ آهن":         dict(ime_symbol="IronOrePlt", cert_qty=1.0, cert_unit="kg", purity=None),
    "میلگرد":                dict(ime_symbol="SteelRebar", cert_qty=1.0, cert_unit="kg", purity=None),
    "گواهی سپرده شمش نقره":  dict(ime_symbol="SilverBar",  cert_qty=1.0, cert_unit="g",  purity=0.9999),   # عیار ۹۹۹٫۹
    "گواهی سپرده شمش طلا":   dict(ime_symbol="GoldBar",    cert_qty=0.1, cert_unit="g",  purity=0.995),    # حداقل ۹۹۵
    "گواهی سپرده سکه طلا":   dict(ime_symbol="GoldCoin",    cert_qty=1, cert_unit="g",  purity=0.995),
}

# ۴) جدول ضرایب
rows = []
for name, spec in CERT_SPEC.items():
    g_unit = GLOBAL_SPEC[name]["unit"]
    base_unit, per_base = GLOBAL_UNIT_PER_BASE[g_unit]
    assert base_unit == spec["cert_unit"], f"{name}: واحد گواهی با واحد جهانی سازگار نیست"
    rows.append(dict(
        name=name,
        ime_symbol=spec["ime_symbol"],
        cert_qty=spec["cert_qty"],
        cert_unit=spec["cert_unit"],
        purity=spec["purity"],
        global_unit=g_unit,
        global_units_per_base=per_base,
        global_units_per_cert=spec["cert_qty"] * per_base,
    ))
FACTORS = pd.DataFrame(rows)
FACTORS

,name,ime_symbol,cert_qty,cert_unit,purity,global_unit,global_units_per_base,global_units_per_cert
0,مس کاتد,CopperCthd,1.0,kg,NaN,USD/lb,2.204623,2.204623
1,شمش روی,ZincIngot,1.0,kg,NaN,USD/t,0.001000,0.001000
2,گندله سنگ آهن,IronOrePlt,1.0,kg,NaN,USD/t,0.001000,0.001000
3,میلگرد,SteelRebar,1.0,kg,NaN,USD/t,0.001000,0.001000
4,گواهی سپرده شمش نقره,SilverBar,1.0,g,0.9999,USD/oz,0.032151,0.032151
5,گواهی سپرده شمش طلا,GoldBar,0.1,g,0.9950,USD/oz,0.032151,0.003215
6,گواهی سپرده سکه طلا,GoldCoin,0.1,g,0.9950,USD/oz,0.032151,0.003215


In [10]:
# پنل دیتا: هر ردیف = (نماد، روز) با قیمت‌های پایان روز مبنا، قیمت جهانی (اسپات/نزدیک‌ترین فیوچرز)،
# فیوچرز یک‌ساله، ضرایب تبدیل و قیمت دلار همان روز

START_DATE = "2025-11-01"   # شروع بازه پنل (نوامبر سال پیش) تا آخرین روز موجود؛ None = کل تاریخچه
GLOBAL_LAG_DAYS = 1      # جلسه بورس کالا قبل از بسته‌شدن بازار جهانیِ همان تاریخ است؛ پس آخرین قیمت جهانی ≤ (روز − ۱)
GLOBAL_TOLERANCE = pd.Timedelta(days=7)
DOLLAR = "دلار"

# ۱) اصلاح قیمت پایانی خراب مبنا: close_price با real_close_price بیش از ۲۵٪ فاصله دارد
#    (مثلاً طلا در ۲۰۲۵/۰۴/۰۸ تا ۰۴/۱۰ ده‌برابر کوچک ثبت شده) -> جایگزینی با real_close_price
m_all = mabna.copy()
bad = (m_all["mabna_real_close_price"] > 0) & \
      ((m_all["mabna_close_price"] / m_all["mabna_real_close_price"] - 1).abs() > 0.25)
m_all["close_fixed"] = bad
m_all.loc[bad, "mabna_close_price"] = m_all.loc[bad, "mabna_real_close_price"]
print("ردیف‌های اصلاح‌شده close_price:", int(bad.sum()))
print(m_all.loc[bad, ["name", "j_date", "mabna_close_price"]].to_string(index=False) if bad.any() else "")

# بازه زمانی پنل (دلار و دارایی‌ها هر دو)
if START_DATE:
    m_all = m_all[m_all["date"] >= pd.Timestamp(START_DATE)]
print("بازه پنل:", m_all["date"].min().date(), "تا", m_all["date"].max().date())

# ۲) دلار: جدا از ردیف‌ها و فقط با تطبیق دقیق تاریخ (بدون پرکردن با روزهای قبل)
dollar = (m_all.loc[m_all["name"] == DOLLAR, ["date", "mabna_close_price"]]
          .rename(columns={"mabna_close_price": "dollar"}).sort_values("date"))
dollar["dollar_same_as_prev"] = dollar["dollar"].diff() == 0       # دلار تکراری (احتمالاً به‌روز نشده)
items = m_all[m_all["name"] != DOLLAR]

# ۳) قیمت جهانی (اسپات یا نزدیک‌ترین فیوچرز) و فیوچرز ۱ ساله، هر دو با تأخیر GLOBAL_LAG_DAYS
#    global_date / global_fut_1y_date = تاریخ واقعی قیمت جهانیِ استفاده‌شده
parts = []
for name, m in items.groupby("name", sort=False):
    m = m.sort_values("date").assign(_key=lambda d: d["date"] - pd.Timedelta(days=GLOBAL_LAG_DAYS))
    g = (global_df[global_df["name"] == name].drop(columns="name")
         .assign(global_date=lambda d: d["date"]).sort_values("date"))
    if not g.empty:
        m = pd.merge_asof(m, g.rename(columns={"date": "_gdate"}), left_on="_key", right_on="_gdate",
                          direction="backward", tolerance=GLOBAL_TOLERANCE).drop(columns="_gdate")
    f = (fut1y_df[fut1y_df["name"] == name].drop(columns="name")
         .assign(global_fut_1y_date=lambda d: d["date"]).sort_values("date"))
    if not f.empty:
        m = pd.merge_asof(m, f.rename(columns={"date": "_fdate"}), left_on="_key", right_on="_fdate",
                          direction="backward", tolerance=GLOBAL_TOLERANCE).drop(columns="_fdate")
    parts.append(m.drop(columns="_key"))

panel = pd.concat(parts, ignore_index=True)
for c in ["global_price", "global_fut_1y"]:        # اگر برای هیچ نمادی نبود، ستون خالی بساز
    if c not in panel.columns:
        panel[c] = np.nan
panel = panel.merge(dollar, on="date", how="left")             # دلار فقط در همان روز
panel = (panel
         .merge(FACTORS[["name", "cert_qty", "cert_unit", "purity", "global_units_per_cert"]],
                on="name", how="left")
         .sort_values(["name", "date"]).reset_index(drop=True))

# شیب ساختار زمانی: فیوچرز ۱ ساله نسبت به قیمت جهانی روز (۱۲ ماهه ≈ سالانه)؛ مبنای افق برای مدل مرحله بعد
panel["futures_basis_1y"] = panel["global_fut_1y"] / panel["global_price"] - 1

print("ردیف‌های بدون دلار:", int(panel["dollar"].isna().sum()), "| بدون قیمت جهانی:", int(panel["global_price"].isna().sum()),
      "| بدون فیوچرز ۱ ساله:", int(panel["global_fut_1y"].isna().sum()))
panel.to_csv("panel.csv", index=False, encoding="utf-8-sig")
panel.tail()

ردیف‌های اصلاح‌شده close_price: 3
               name     j_date  mabna_close_price
گواهی سپرده شمش طلا 1404/01/19         10227520.0
گواهی سپرده شمش طلا 1404/01/20          9881000.0
گواهی سپرده شمش طلا 1404/01/21         10198980.0
بازه پنل: 2025-11-01 تا 2026-10-18
ردیف‌های بدون دلار: 78 | بدون قیمت جهانی: 78 | بدون فیوچرز ۱ ساله: 78


,name,code,mabna_open_price,mabna_high_price,mabna_low_price,mabna_close_price,mabna_real_close_price,mabna_volume,mabna_value,mabna_trade_count,...,global_fut_1y,global_fut_1y_source,global_fut_1y_date,dollar,dollar_same_as_prev,cert_qty,cert_unit,purity,global_units_per_cert,futures_basis_1y
1417,گواهی سپرده شمش نقره,SilverBar,5040000.0,5089900.0,4850000.0,4975266.0,4862100.0,400137.0,1.990788e+12,3083.0,...,62.947803,SI 365d constant-maturity (23 قرارداد Yahoo/CMX),2026-10-12,2577100.0,False,1.0,g,0.9999,0.032151,0.046670
1418,گواهی سپرده شمش نقره,SilverBar,4989000.0,5159990.0,4970900.0,5076112.0,5039090.0,531900.0,2.699984e+12,3137.0,...,61.949508,SI 365d constant-maturity (23 قرارداد Yahoo/CMX),2026-10-13,2639000.0,False,1.0,g,0.9999,0.032151,0.047134
1419,گواهی سپرده شمش نقره,SilverBar,5091900.0,5091900.0,5040050.0,5059610.0,5046500.0,59789.0,3.025091e+11,826.0,...,58.230033,SI 365d constant-maturity (23 قرارداد Yahoo/CMX),2026-10-14,2679000.0,False,1.0,g,0.9999,0.032151,0.048245
1420,گواهی سپرده شمش نقره,SilverBar,5030500.0,5030500.0,4890600.0,4940323.0,4939990.0,289880.0,1.432101e+12,2724.0,...,59.700180,SI 365d constant-maturity (23 قرارداد Yahoo/CMX),2026-10-16,2638000.0,False,1.0,g,0.9999,0.032151,0.041797
1421,گواهی سپرده شمش نقره,SilverBar,4900000.0,4945000.0,4798720.0,4840249.0,4818000.0,119524.0,5.785259e+11,1241.0,...,59.700180,SI 365d constant-maturity (23 قرارداد Yahoo/CMX),2026-10-16,2608000.0,False,1.0,g,0.9999,0.032151,0.041797


In [11]:
import numpy as np

panel["implied_dollar"] = panel["mabna_close_price"] / (panel["global_price"] * panel["global_units_per_cert"])
panel["implied_dollar_bubble"] = (panel["implied_dollar"] / panel["dollar"]) - 1

# ---- بررسی سلامت واحد/مرجع ----
# دلار ذاتی باید هم‌مرتبه با دلار بازار باشد (نسبت ≈ ۱). نسبت خیلی کم یا زیاد یعنی خطای واحد/ضریب
# (مثلاً گواهی طلا ۰٫۱ گرم است و اگر ۱ گرم فرض شود نسبت ≈ ۰٫۱ می‌شود) یا مرجع جهانی نامناسب (مثل HRC برای میلگرد).
_ok = panel.dropna(subset=["implied_dollar", "dollar"]).sort_values("date")
sanity = (_ok.assign(ratio=_ok["implied_dollar"] / _ok["dollar"])
          .groupby("name")
          .agg(rows=("ratio", "size"), median_ratio=("ratio", "median"), last_ratio=("ratio", "last"),
               last_implied=("implied_dollar", "last"), last_market=("dollar", "last"))
          .round(2))
sanity["flag"] = np.where((sanity["median_ratio"] < 0.4) | (sanity["median_ratio"] > 2.5), "⚠ واحد/مرجع را چک کن", "")
display(sanity)


def _show(fig, name):
    try:
        fig.show()
    except ValueError as e:          # nbformat نصب نیست
        fig.write_html(f"{name}.html"); print(f"نمودار در {name}.html ذخیره شد:", e)


# دلار ذاتی هر نماد در کنار دلار بازار (ریال)
_lines = pd.concat([
    panel[["date", "name", "implied_dollar"]].rename(columns={"implied_dollar": "value"}),
    dollar.assign(name="دلار بازار").rename(columns={"dollar": "value"})[["date", "name", "value"]],
])
_show(px.line(_lines, x="date", y="value", color="name", title="دلار ذاتی هر نماد در برابر دلار بازار (ریال)"), "implied_dollar")
_show(px.line(panel, x="date", y="implied_dollar_bubble", color="name", title="حباب دلار ذاتی هر نماد"), "implied_dollar_bubble")

,rows,median_ratio,last_ratio,last_implied,last_market,flag
name,,,,,,
شمش روی,209,0.88,0.94,2459948.52,2608000.0,
مس کاتد,212,0.89,0.94,2443926.38,2608000.0,
میلگرد,209,0.83,0.97,2518828.69,2608000.0,
گندله سنگ آهن,212,0.50,0.71,1855961.48,2608000.0,
گواهی سپرده شمش طلا,212,0.98,0.97,2526628.18,2608000.0,
گواهی سپرده شمش نقره,212,1.10,1.01,2627145.48,2608000.0,


## شواهد: نوع سیگنال هر دارایی

**سؤال:** آیا قیمت دلاری هر نماد در بورس کالا با مرجع جهانی‌اش هم‌حرکت است؟ اگر نه، «حباب منفی» به‌معنای ارزانی نسبت به جهان نیست و بازگشت آن به میانگین، بیشتر **بازگشت قیمت دلاری خود نماد به میانگین خودش** است (`self-referenced`).

| نماد | برچسب |
|---|---|
| طلا (`GC=F`)، مس (`HG=F`) | `global-anchored` |
| میلگرد (SHFE `rb_f`)، روی (SHFE `zn_f`)، گندله (SGX `M65F`) | `self-referenced` |
| نقره (`SI=F`) | `global-anchored` (همبستگی ۰٫۴۷ / ۰٫۶۸ / ۰٫۷۹) |

**خلاصه شواهد** (جزئیات کامل، جدول ۱۵ مرجع و محدودیت‌ها: `docs/signal_evidence.md`)
- همبستگی تغییر قیمت دلاری بورس کالا با مرجع پنل در ۱ / ۵ / ۱۰ جلسه: طلا ۰٫۴۸ / ۰٫۷۷ / ۰٫۷۵؛ مس ۰٫۳۱ / ۰٫۲۵ / ۰٫۵۶؛ میلگرد −۰٫۰۷ / −۰٫۱۲ / −۰٫۲۴؛ روی ۰٫۰۴ / ۰٫۰۶ / ۰٫۰۳؛ گندله −۰٫۰۷ / −۰٫۰۱ / ۰٫۰۴؛ نقره ۰٫۴۷ / ۰٫۶۸ / ۰٫۷۹.
- غربال ۱۵ مرجع جایگزین (SHFE، Yahoo، SGX): برای میلگرد بهتر از `rb_f`، آهن SGX ۶۵٪ (۰٫۰۶ / ۰٫۳۷ / ۰٫۴۶)؛ برای روی مس `HG=F` (۰٫۲۰ / ۰٫۱۹ / ۰٫۶۰)؛ برای گندله هیچ مرجع آهنی ارتباطی نداشت.
- همه‌ی نمادها بازگشت به میانگین حباب نشان می‌دهند (همبستگی انحراف با تغییر ۱۰ جلسه بعد: طلا −۰٫۷۳، مس −۰٫۵۳، نقره −۰٫۴۵، میلگرد −۰٫۵۰، روی −۰٫۳۶، گندله −۰٫۱۳).
- **احتیاط:** حدود ۲۴۰ روز؛ ۲۴ مشاهده‌ی ۱۰ جلسه‌ای؛ ۱۵ مرجع تست شده (خطای مقایسه چندگانه)؛ مشاهدات بازگشت هم‌پوشان‌اند. بنابراین مرجع‌ها بر اساس غربال عوض نشدند و برچسب فقط توضیحی است و وزن‌دهی مدل را تغییر نمی‌دهد.

سلول بعد همین آمار را (با مرجع‌های فعلی پنل) دوباره می‌سازد.

In [12]:
import numpy as np

# برچسب نوع سیگنال هر دارایی (توضیحی؛ وزن‌دهی مدل را تغییر نمی‌دهد). شواهد: docs/signal_evidence.md
SIGNAL_TYPE = {
    "گواهی سپرده شمش طلا":  "global-anchored",
    "مس کاتد":              "global-anchored",
    "میلگرد":               "self-referenced",
    "شمش روی":              "self-referenced",
    "گندله سنگ آهن":        "self-referenced",
    "گواهی سپرده شمش نقره": "global-anchored",
}


def _corr(a, b, min_n=8):
    d = pd.concat([a, b], axis=1).dropna()
    return d.iloc[:, 0].corr(d.iloc[:, 1]) if len(d) >= min_n else np.nan


ev = panel[panel["dollar"].notna() & panel["global_price"].notna()].copy()
ev["ime_usd"] = ev["mabna_close_price"] / ev["dollar"]               # قیمت دلاری بورس کالا (دلار به ازای هر گواهی)
ev["bubble"] = ev["mabna_close_price"] / (ev["global_price"] * ev["global_units_per_cert"]) / ev["dollar"] - 1

rows = []
for name, g in ev.groupby("name"):
    g = g.sort_values("date").reset_index(drop=True)
    r = {"name": name, "signal_type": SIGNAL_TYPE.get(name, "-"), "n": len(g)}
    for step in (1, 5, 10):                                         # مشاهدات غیرهم‌پوشان
        a = np.log(g["ime_usd"]).iloc[::step].diff()
        b = np.log(g["global_price"]).iloc[::step].diff()
        r[f"corr_global_{step}"] = _corr(a, b)
    dev = g["bubble"] - g["bubble"].expanding(min_periods=40).mean()      # انحراف از میانگین گذشته
    for h in (5, 10):
        r[f"revert_corr_{h}"] = _corr(dev, g["bubble"].shift(-h) - g["bubble"])   # منفی = بازگشت
    rows.append(r)
evidence = pd.DataFrame(rows).set_index("name").round(2)

# همبستگی دارایی‌های بورس کالا با هم (قیمت دلاری، ۵ جلسه)
wide = ev.pivot(index="date", columns="name", values="ime_usd").sort_index()
cross = np.log(wide).iloc[::5].diff().corr().round(2)

print("هم‌حرکتی با مرجع جهانی (۱/۵/۱۰ جلسه) و بازگشت به میانگین (revert_corr؛ منفی = بازگشت):")
display(evidence)
print("\nهمبستگی بین خود دارایی‌ها (۵ جلسه):")
display(cross)

هم‌حرکتی با مرجع جهانی (۱/۵/۱۰ جلسه) و بازگشت به میانگین (revert_corr؛ منفی = بازگشت):


,signal_type,n,corr_global_1,corr_global_5,corr_global_10,revert_corr_5,revert_corr_10
name,,,,,,,
شمش روی,self-referenced,209,0.04,0.06,0.03,-0.30,-0.36
مس کاتد,global-anchored,212,0.31,0.25,0.56,-0.41,-0.53
میلگرد,self-referenced,209,-0.07,-0.12,-0.24,-0.33,-0.47
گندله سنگ آهن,self-referenced,212,-0.07,-0.01,0.04,-0.10,-0.13
گواهی سپرده شمش طلا,global-anchored,212,0.48,0.77,0.75,-0.64,-0.73
گواهی سپرده شمش نقره,global-anchored,212,0.47,0.68,0.79,-0.39,-0.45



همبستگی بین خود دارایی‌ها (۵ جلسه):


name,شمش روی,مس کاتد,میلگرد,گندله سنگ آهن,گواهی سپرده شمش طلا,گواهی سپرده شمش نقره
name,,,,,,
شمش روی,1.00,0.31,0.21,0.33,0.20,0.45
مس کاتد,0.31,1.00,0.33,0.18,-0.12,0.23
میلگرد,0.21,0.33,1.00,0.61,-0.21,-0.04
گندله سنگ آهن,0.33,0.18,0.61,1.00,-0.04,0.16
گواهی سپرده شمش طلا,0.20,-0.12,-0.21,-0.04,1.00,0.68
گواهی سپرده شمش نقره,0.45,0.23,-0.04,0.16,0.68,1.00


## مدل اولیه وزن‌دهی بر اساس حباب منفی

**ایده:** حباب منفی یعنی دارایی در بورس کالا نسبت به دلار ذاتی ارزان‌تر از دلار بازار است. انتظار می‌رود این شکاف به‌مرور برگردد (mean reversion). **دلار فقط مبنای سنجش حباب است و جزو دارایی‌های پرتفوی نیست.**

**پشتوانه ادبیات** (خلاصه جستجو؛ جزئیات دقیق هر مقاله را قبل از استناد رسمی از خود مقاله چک کن)
- **تخفیف صندوق‌های closed-end:** Pontiff (1995, JFE) نشان داد خرید صندوق‌های با تخفیف عمیق بازدهی بالاتری دارد و منشأ آن برگشت تخفیف به میانگین است. استراتژی عملی: خرید وقتی تخفیف نسبت به **میانگین تاریخی خودش** عمیق است. هشدار: هزینه معامله می‌تواند سود را حذف کند و میانگین ممکن است تغییر کند.
- **Pairs trading:** Gatev, Goetzmann & Rouwenhorst (2006, RFS)؛ ورود حدوداً با انحراف ۲ انحراف‌معیاری (طبق خلاصه‌ی ثانویه؛ از خود مقاله تأیید کن).
- **زمان‌بندی بهینه روی فرایند OU:** Leung & Li (2015) و Bertram: معامله‌ی بهینه‌ی یک اسپرد mean-reverting ناحیه‌ی ورود/خروج دارد؛ ورودی اصلی سرعت برگشت و نوسان است.
- **اندازه پوزیشن:** Moskowitz, Ooi & Pedersen (2012) پوزیشن‌ها را با نوسان گذشته مقیاس می‌دهند؛ Kelly کسری (fractional) برای کاهش خطای برآورد.
- **ساده‌سازی در برابر بهینه‌سازی:** DeMiguel, Garlappi & Uppal (2009, RFS): بهینه‌سازی با میانگین/کوواریانس برآوردشده معمولاً از 1/N بهتر نیست. بنابراین مدل زیر عمداً ساده است و وزن را فقط حول وزن پایه (معکوس نوسان) کج می‌کند.

**مدل (روزانه، بدون نگاه به آینده؛ پرتفوی فقط دارایی‌های کالایی و کاملاً سرمایه‌گذاری‌شده)**
1. `حباب_t` = دلار ذاتی ÷ دلار بازار − ۱؛ `میانگین_t` و `انحراف_t` = میانگین و انحراف‌معیار حباب همان دارایی از ابتدا **تا همان روز**.
2. شاخص ارزانی نسبی: `z_t = (میانگین_t − حباب_t) / انحراف_t` (مثبت یعنی ارزان‌تر از معمول خودش).
3. فقط وقتی حباب منفی است (`حباب_t < 0`) سیگنال داریم: `قدرت_t = min(max(z_t, 0), Z_CAP) / Z_CAP` (بین ۰ و ۱).
4. وزن پایه هر دارایی ∝ ۱/نوسان. وزن نهایی: `وزن_i ∝ پایه_i × (FLOOR + قدرت_i)`، نرمال‌شده به مجموع ۱. پس دارایی‌هایی که الان نسبت به خودشان ارزان‌ترند وزن بیشتری می‌گیرند و بدون هیچ سیگنالی وزن‌ها همان وزن پایه است. فروش استقراضی نداریم.
5. وزن‌های روز t با `LAG` روز تأخیر اعمال می‌شوند و هزینه معامله کسر می‌شود.

**بازدهی «بر حسب واحد دارایی»:** ارزش پرتفوی ÷ قیمت دارایی i، نسبت به شروع. مقدار `۱` یعنی برابر نگه‌داشتن ۱ واحد آن دارایی؛ `۱٫۲` یعنی ۲۰٪ واحد بیشتر. همین را برای **بنچمارک** (همان وزن پایه بدون سیگنال) هم حساب می‌کنیم تا افزوده‌ی خودِ سیگنال معلوم شود (`signal_value_units_%`).

**محدودیت‌ها:** سطح حباب شامل پایه‌ی قیمتی پروکسی‌ها (میلگرد، گندله)، تفاوت کیفیت و عیار است؛ برای همین سیگنال بر اساس انحراف از **میانگین خود دارایی** است نه سطح مطلق. تغییر ساختاری (مثل شوک‌های ارزی) می‌تواند میانگین را جابه‌جا کند. پارامترها اعتبار بیرون‌نمونه‌ای ندارند.

**نوع سیگنال هر دارایی (برچسب توضیحی):** طلا، مس و نقره `global-anchored` (قیمت دلاری‌شان با مرجع جهانی هم‌حرکت است)؛ میلگرد، روی و گندله `self-referenced` (همبستگی با مرجع جهانی ناچیز؛ سیگنال عملاً بازگشت قیمت دلاری خود نماد به میانگینش است). این برچسب وزن‌دهی را تغییر نمی‌دهد و در `summary` ستون `signal_type` است. شواهد: سلول «شواهد» بالاتر و `docs/signal_evidence.md`.


In [13]:
import numpy as np

# ===================== پارامترها =====================
MIN_HIST   = 60     # حداقل روز سابقه برای میانگین/انحراف گذشته (قبل از آن سیگنال صفر است)
Z_CAP      = 2.0    # z ≥ Z_CAP => قدرت سیگنال ۱؛ الهام از آستانه ۲ انحراف‌معیار
FLOOR      = 0.25   # حداقل ضریب هر دارایی در وزن‌دهی: وزن ∝ پایه × (FLOOR + قدرت).  FLOOR=0 یعنی فقط دارایی‌های با سیگنال
VOL_WINDOW = 60     # پنجره نوسان روزانه برای وزن پایه معکوس نوسان
INV_VOL    = True   # True: وزن پایه ∝ ۱/نوسان ؛ False: برابر (1/N)
COST_BPS   = 20     # هزینه هر معامله (bps از مبلغ معامله‌شده)
LAG        = 1      # وزن تصمیم‌گرفته‌شده در پایان روز t از روز t+LAG اعمال می‌شود

# دلار فقط مبنای محاسبه حباب است؛ جزو دارایی‌های پرتفوی نیست.
# پرتفوی فقط از دارایی‌های کالایی ساخته می‌شود و همیشه کاملاً سرمایه‌گذاری‌شده است (مجموع وزن‌ها = ۱).

# ===================== ورودی‌ها =====================
panel["implied_dollar"] = panel["mabna_close_price"] / (panel["global_price"] * panel["global_units_per_cert"])
panel["implied_dollar_bubble"] = panel["implied_dollar"] / panel["dollar"] - 1
panel["implied_dollar_bubble"] = panel["implied_dollar_bubble"].replace([np.inf, -np.inf], np.nan)

bub = panel.pivot(index="date", columns="name", values="implied_dollar_bubble").sort_index()
px_ = panel.pivot(index="date", columns="name", values="mabna_close_price").sort_index()
usd = panel.groupby("date")["dollar"].first().sort_index()

assets = [c for c in bub.columns if bub[c].notna().sum() > MIN_HIST + 30]
print("دارایی‌های مدل:", assets)
print("خارج از مدل (داده کافی ندارند):", [c for c in bub.columns if c not in assets])

cal = usd.dropna().index          # تقویم = روزهایی که حباب قابل‌محاسبه است (دلار موجود است)


def to_cal(df, limit=5):
    """روی تقویم بیاور؛ روزهای بدون معامله آن دارایی با آخرین مقدار (حداکثر limit روز) پر می‌شود"""
    return df.reindex(df.index.union(cal)).sort_index().ffill(limit=limit).reindex(cal)


# ===================== سیگنال (فقط با اطلاعات تا همان روز) =====================
b = bub[assets]
mean_b = b.expanding(min_periods=MIN_HIST).mean()          # میانگین حباب از گذشته تا امروز
std_b  = b.expanding(min_periods=MIN_HIST).std()
z = (mean_b - b) / std_b.replace(0, np.nan)                # مثبت = ارزان‌تر از معمول خودش
strength = (z.clip(lower=0, upper=Z_CAP) / Z_CAP).where(b < 0, 0.0)   # فقط وقتی حباب منفی است
strength = to_cal(strength).fillna(0.0)

# ===================== وزن‌ها =====================
P  = to_cal(px_[assets])
R  = P.pct_change()
vol = R.rolling(VOL_WINDOW, min_periods=20).std()
avail = P.notna() & vol.notna()
iv = (1.0 / vol).where(avail) if INV_VOL else avail.astype(float).where(avail)
base = iv.div(iv.sum(axis=1), axis=0).fillna(0.0)          # وزن پایه بدون سیگنال (مجموع = ۱)

tilt = base * (FLOOR + strength)                           # سیگنال وزن را به سمت دارایی‌های ارزان‌تر می‌کشد
wsum = tilt.sum(axis=1)
weights_target = tilt.div(wsum.where(wsum > 0), axis=0)
weights_target = weights_target.fillna(base).fillna(0.0)   # اگر مجموع صفر شد (FLOOR=0 و بدون سیگنال): وزن پایه

# شروع مشترک: اولین روزی که برای همه دارایی‌ها سابقه کافی داریم
ready = std_b.notna().all(axis=1).astype(float)
ready = ready.reindex(ready.index.union(cal)).sort_index().ffill().reindex(cal).fillna(0) > 0
t0 = ready[ready].index[0]
print("شروع بک‌تست:", t0.date(), "| تعداد روز:", int((cal >= t0).sum()))

weights_target.loc[t0:].tail()

دارایی‌های مدل: ['شمش روی', 'مس کاتد', 'میلگرد', 'گندله سنگ آهن', 'گواهی سپرده شمش طلا', 'گواهی سپرده شمش نقره']
خارج از مدل (داده کافی ندارند): ['شمش سرب']
شروع بک‌تست: 2026-02-08 | تعداد روز: 153


name,شمش روی,مس کاتد,میلگرد,گندله سنگ آهن,گواهی سپرده شمش طلا,گواهی سپرده شمش نقره
date,,,,,,
2026-10-13,0.097336,0.106169,0.100368,0.081799,0.252347,0.361981
2026-10-14,0.152605,0.166323,0.159274,0.129171,0.235593,0.157035
2026-10-15,0.157570,0.171537,0.163858,0.133529,0.208376,0.165130
2026-10-17,0.130726,0.143763,0.150886,0.112249,0.319331,0.143045
2026-10-18,0.127057,0.139726,0.144454,0.109137,0.339959,0.139668


In [14]:
# ===================== بک‌تست =====================
sel = cal[cal >= t0]                                       # روزهای بک‌تست (از t0)
cost_rate = COST_BPS / 1e4


def backtest(w_target):
    """پرتفوی فقط از دارایی‌ها؛ هر روز به وزن هدف (با LAG روز تأخیر) بالانس می‌شود.
    خروجی: ارزش پرتفوی (۱ در شروع)، وزن‌های اعمال‌شده، گردش روزانه"""
    w_ex = w_target.shift(LAG).fillna(0.0)
    turn = w_ex.diff().abs().sum(axis=1).fillna(0.0)
    ret = (w_ex * R.fillna(0.0)).sum(axis=1) - cost_rate * turn
    v = (1.0 + ret.loc[sel[1:]]).cumprod()
    v = pd.concat([pd.Series(1.0, index=[t0]), v])
    return v, w_ex, turn


V,   w_exec,  turnover   = backtest(weights_target)        # با سیگنال حباب
V_b, w_exec_b, turnover_b = backtest(base)                 # بنچمارک: همان وزن پایه بدون سیگنال

# --- بازدهی بر حسب واحد هر دارایی ---
# تعداد واحد دارایی i که ارزش پرتفوی می‌خرد، نسبت به شروع:  V_t × P_i(t0) / P_i(t)
# مقدار ۱ = برابر نگه‌داشتن ۱ واحد (با همان ارزش اولیه)؛ >۱ یعنی واحد بیشتر
def in_units(v):
    Pt = P.loc[v.index, assets]
    return Pt.rdiv(v, axis=0).mul(P.loc[t0, assets], axis=1)


units_portfolio = in_units(V)
units_benchmark = in_units(V_b)

# ===================== خلاصه =====================
years = (sel[-1] - sel[0]).days / 365.25


def max_dd(v):
    return (v / v.cummax() - 1.0).min()


summary = pd.DataFrame({
    "avg_weight": w_exec.loc[sel].mean(),
    "avg_weight_benchmark": w_exec_b.loc[sel].mean(),
    "final_units": units_portfolio.iloc[-1],                         # شروع = ۱
    "excess_units_%": (units_portfolio.iloc[-1] - 1) * 100,          # نسبت به نگه‌داشتن ۱ واحد این دارایی
    "final_units_benchmark": units_benchmark.iloc[-1],
    "excess_units_benchmark_%": (units_benchmark.iloc[-1] - 1) * 100,
    "signal_value_units_%": (units_portfolio.iloc[-1] - units_benchmark.iloc[-1]) * 100,   # افزوده‌ی سیگنال نسبت به بنچمارک
    "maxdd_units_%": units_portfolio.apply(max_dd) * 100,
})
print(f"دوره: {sel[0].date()} تا {sel[-1].date()} ({years:.2f} سال) | هزینه {COST_BPS} bps | تأخیر {LAG} روز")
print(f"پرتفوی (ریال):   بازده کل {V.iloc[-1]-1:.1%} | سالانه {V.iloc[-1]**(1/years)-1:.1%} | حداکثر افت {max_dd(V):.1%}")
print(f"بنچمارک بدون سیگنال: بازده کل {V_b.iloc[-1]-1:.1%} | سالانه {V_b.iloc[-1]**(1/years)-1:.1%} | حداکثر افت {max_dd(V_b):.1%}")
print(f"میانگین گردش روزانه: {turnover.loc[sel].mean():.3f} (بنچمارک {turnover_b.loc[sel].mean():.3f})")

weights = w_exec.loc[sel]
weights.to_csv("weights.csv", encoding="utf-8-sig")
summary.to_csv("summary.csv", encoding="utf-8-sig")


def long(df, value):
    """جدول عریض (ایندکس=تاریخ) -> بلند برای plotly"""
    return df.rename_axis("date").reset_index().melt("date", var_name="دارایی", value_name=value)


def show(fig, name):
    try:
        fig.show()
    except ValueError as e:        # nbformat نصب نیست
        fig.write_html(f"{name}.html"); print(f"نمودار در {name}.html ذخیره شد:", e)


show(px.area(long(weights, "وزن"), x="date", y="وزن", color="دارایی",
             title="وزن دارایی‌ها در پرتفوی"), "weights")
show(px.line(long(units_portfolio - 1, "واحد اضافه"), x="date", y="واحد اضافه", color="دارایی",
             title="پرتفوی بر حسب واحد هر دارایی (۰ = برابر نگه‌داشتن ۱ واحد)"), "units_portfolio")
show(px.line(long(units_portfolio - units_benchmark, "واحد اضافه نسبت به بنچمارک"), x="date",
             y="واحد اضافه نسبت به بنچمارک", color="دارایی",
             title="افزوده‌ی سیگنال: پرتفوی − بنچمارک (بر حسب واحد هر دارایی)"), "signal_value")
summary.round(3)

دوره: 2026-02-08 تا 2026-10-18 (0.69 سال) | هزینه 20 bps | تأخیر 1 روز
پرتفوی (ریال):   بازده کل 88.4% | سالانه 150.4% | حداکثر افت -14.7%
بنچمارک بدون سیگنال: بازده کل 79.6% | سالانه 133.6% | حداکثر افت -15.4%
میانگین گردش روزانه: 0.164 (بنچمارک 0.013)


,avg_weight,avg_weight_benchmark,final_units,excess_units_%,final_units_benchmark,excess_units_benchmark_%,signal_value_units_%,maxdd_units_%
name,,,,,,,,
شمش روی,0.162,0.179,0.919,-8.105,0.876,-12.413,4.308,-22.236
مس کاتد,0.180,0.167,0.964,-3.573,0.919,-8.093,4.520,-24.129
میلگرد,0.149,0.158,0.867,-13.328,0.826,-17.391,4.063,-27.654
گندله سنگ آهن,0.218,0.207,0.920,-7.976,0.877,-12.290,4.314,-24.648
گواهی سپرده شمش طلا,0.210,0.171,1.351,35.126,1.288,28.792,6.335,-8.504
گواهی سپرده شمش نقره,0.081,0.119,2.077,107.727,1.980,97.989,9.738,-15.691


In [15]:
# برچسب نوع سیگنال را به خلاصه اضافه کن (توضیحی؛ شواهد: docs/signal_evidence.md)
summary["signal_type"] = summary.index.map(SIGNAL_TYPE)
summary = summary[["signal_type"] + [c for c in summary.columns if c != "signal_type"]]
summary.to_csv("summary.csv", encoding="utf-8-sig")
summary.round(3)

,signal_type,avg_weight,avg_weight_benchmark,final_units,excess_units_%,final_units_benchmark,excess_units_benchmark_%,signal_value_units_%,maxdd_units_%
name,,,,,,,,,
شمش روی,self-referenced,0.162,0.179,0.919,-8.105,0.876,-12.413,4.308,-22.236
مس کاتد,global-anchored,0.180,0.167,0.964,-3.573,0.919,-8.093,4.520,-24.129
میلگرد,self-referenced,0.149,0.158,0.867,-13.328,0.826,-17.391,4.063,-27.654
گندله سنگ آهن,self-referenced,0.218,0.207,0.920,-7.976,0.877,-12.290,4.314,-24.648
گواهی سپرده شمش طلا,global-anchored,0.210,0.171,1.351,35.126,1.288,28.792,6.335,-8.504
گواهی سپرده شمش نقره,global-anchored,0.081,0.119,2.077,107.727,1.980,97.989,9.738,-15.691
